In [25]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql import types
from pyspark.sql import functions as F

In [26]:
spark = SparkSession.builder \
    .master("local[*]") \
    .appName('homework_2026') \
    .getOrCreate()

26/03/07 17:42:49 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


## Q1: Spark Version

In [27]:
spark.version

'4.1.1'

## Q2: Yellow November 2025

Read the November 2025 Yellow into a Spark DataFrame.
Repartition to 4 partitions and save to parquet.
What is the average size of the `.parquet` files?

In [28]:
!wget -O yellow_tripdata_2025-11.parquet https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet

--2026-03-07 17:42:55--  https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 2600:9000:2684:ac00:b:20a5:b140:21, 2600:9000:2684:c600:b:20a5:b140:21, 2600:9000:2684:d400:b:20a5:b140:21, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|2600:9000:2684:ac00:b:20a5:b140:21|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 71134255 (68M) [binary/octet-stream]
Saving to: ‘yellow_tripdata_2025-11.parquet’

yellow_tripdata_202 100%[===================>]  67.84M  82.3MB/s    in 0.8s    

2026-03-07 17:42:56 (82.3 MB/s) - ‘yellow_tripdata_2025-11.parquet’ saved [71134255/71134255]



In [29]:
df = spark.read.parquet('yellow_tripdata_2025-11.parquet')
df.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)



In [30]:
df = df.repartition(4)
df.write.parquet('data/pq/yellow/2025/11/', mode='overwrite')

In [31]:
!ls -lh data/pq/yellow/2025/11/*.parquet

-rw-r--r--@ 1 ettyekhon  staff    24M  7 Mar 17:43 data/pq/yellow/2025/11/part-00000-21ec54fc-3f51-473c-8dbf-a9a55f67a00d-c000.snappy.parquet
-rw-r--r--@ 1 ettyekhon  staff    24M  7 Mar 17:43 data/pq/yellow/2025/11/part-00001-21ec54fc-3f51-473c-8dbf-a9a55f67a00d-c000.snappy.parquet
-rw-r--r--@ 1 ettyekhon  staff    24M  7 Mar 17:43 data/pq/yellow/2025/11/part-00002-21ec54fc-3f51-473c-8dbf-a9a55f67a00d-c000.snappy.parquet
-rw-r--r--@ 1 ettyekhon  staff    24M  7 Mar 17:43 data/pq/yellow/2025/11/part-00003-21ec54fc-3f51-473c-8dbf-a9a55f67a00d-c000.snappy.parquet


## Q3: Count Records

How many taxi trips were there on the 15th of November?
Consider only trips that **started** on the 15th.

In [32]:
df.withColumn('pickup_date', F.to_date(df.tpep_pickup_datetime)) \
    .filter("pickup_date = '2025-11-15'") \
    .count()

162604

## Q4: Longest Trip

What is the length of the longest trip in the dataset **in hours**?

In [33]:
df.withColumn(
    'duration_hours',
    (F.unix_timestamp('tpep_dropoff_datetime') - F.unix_timestamp('tpep_pickup_datetime')) / 3600
) \
    .select(F.max('duration_hours').alias('max_duration_hours')) \
    .show()

[Stage 48:===================================================>    (11 + 1) / 12]

+------------------+
|max_duration_hours|
+------------------+
| 90.64666666666666|
+------------------+



## Q5: User Interface

Spark's User Interface which shows the application's dashboard runs on which local port?

**Answer: 4040**

## Q6: Least Frequent Pickup Location Zone

Using the zone lookup data and the Yellow November 2025 data,
what is the name of the **LEAST** frequent pickup location Zone?

In [34]:
!wget -O taxi_zone_lookup.csv https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv

--2026-03-07 17:48:43--  https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 2600:9000:2684:1400:b:20a5:b140:21, 2600:9000:2684:be00:b:20a5:b140:21, 2600:9000:2684:2e00:b:20a5:b140:21, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|2600:9000:2684:1400:b:20a5:b140:21|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 12331 (12K) [text/csv]
Saving to: ‘taxi_zone_lookup.csv’

taxi_zone_lookup.cs 100%[===================>]  12.04K  --.-KB/s    in 0s      

2026-03-07 17:48:43 (1.44 GB/s) - ‘taxi_zone_lookup.csv’ saved [12331/12331]



In [35]:
df_zones = spark.read \
    .option('header', 'true') \
    .option('inferSchema', 'true') \
    .csv('taxi_zone_lookup.csv')

df.createOrReplaceTempView('yellow_2025_11')
df_zones.createOrReplaceTempView('zones')

In [36]:
spark.sql("""
SELECT
    z.Zone,
    COUNT(1) AS trip_count
FROM
    yellow_2025_11 t
    JOIN zones z ON t.PULocationID = z.LocationID
GROUP BY
    z.Zone
ORDER BY
    trip_count ASC
LIMIT 5;
""").show(truncate=False)

[Stage 56:======================================>                  (8 + 4) / 12]

+---------------------------------------------+----------+
|Zone                                         |trip_count|
+---------------------------------------------+----------+
|Governor's Island/Ellis Island/Liberty Island|1         |
|Eltingville/Annadale/Prince's Bay            |1         |
|Arden Heights                                |1         |
|Port Richmond                                |3         |
|Rikers Island                                |4         |
+---------------------------------------------+----------+



In [37]:
spark.stop()